# import packages


In [1]:
import os
import pandas as pd
import plotly.graph_objects as go
import re 

# Waveform Time Points

In [2]:
order_of_time_points = [
    "0",
    "6",
    "11",
    "16",
    "21",
    "26",
    # "30",
    "31",
    "36",
    "41",
    "46",
    "51",
    "56",
    # '60',
    "61",
    "66",
    "71",
    "76",
    "81", 
    "86",
    # "91"
    "121",
    "181",
    "241"
]

# PV Time Points

In [ ]:
time_point_order = [
    "T0-1",
    "T5-6",
    "T10-11",
    "T15-16",
    "T20-21",
    "T25-26",
    'T29-30',
    # # "T30-T31",
    "T35-36",
    "T40-41",
    "T45-46",
    "T50-51",
    "T55-56",
    "T59-60", 
    # "T60-61",
    "T64-65", 
    # "T65-66", 
    "T70-71", 
    "T74-75",
    "T75-76",
    "T80-81",
    "T84-85" ,
    # "T85-86",
    # "T90-91"
    "T120-121",
    "T180-181",
    "T239-240"
]

# Waveform Parameters

In [4]:
# Define waveform columns to analyze
columns_to_analyze = [
    'Avg',
    'Max',
    'Min',
    'Heart Rate',
    'Pseudo Cardiac Output'
]

# PV Parameters

In [5]:
# List of terms to be plotted
terms_to_plot = [
    "Stroke Work",
    "Stroke Volume", 
    "Ejection Fraction",
    "Heart Rate",
    "Cardiac Output",
    "Elastance",
    "ESV",
    "ESP",
    "EDV",
    "EDP"
]

# Waveform- By Hemorrhage Level

In [6]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ------------------------
# Parameters you can adjust
# ------------------------

file_path = 'Fractional_Increase_0.xlsx'

AXIS_TICK_SIZE = 16
LABEL_SIZE = 18
LEGEND_SIZE = 14

desired_length, desired_width = 30, 8  # inches

# 🔹 NEW: Control x-axis tick spacing here
X_TICK_INTERVAL = 15   # <-- change this to any number of minutes you want

color_palette = ['black', 'blue', 'red', 'purple', 'orange', 'cyan', 'magenta', 'brown', 'pink']

occlusion_markers = {
    'Full Occlusion': 'o',
    'Partial Occlusion': 'o',
    'No Occlusion': 'o'
}

# 🔹 NEW: Forced legend naming
occlusion_label_map = {
    'No': 'No Aortic Occlusion',
    'No Occlusion': 'No Aortic Occlusion',
    'Partial': 'p-REBOA',
    'Partial Occlusion': 'p-REBOA',
    'Full': 'f-REBOA',
    'Full Occlusion': 'f-REBOA'
}

output_dir = 'Hemodynamics'
os.makedirs(output_dir, exist_ok=True)

# ------------------------
# Load and preprocess data
# ------------------------
df = pd.read_excel(file_path)

# Convert time points to integers then shift by -1
order_of_time_points = [int(tp) for tp in order_of_time_points]
df['Time Point'] = df['Time Point'].apply(lambda x: x - 1 if x in order_of_time_points and x != 0 else x)
order_of_time_points = [tp - 1 if tp != 0 else tp for tp in order_of_time_points]
order_of_time_points = [str(tp) for tp in order_of_time_points]

# Create Test Group (Occlusion only here)
df['Test Group'] = df['Occlusion Group']

# Sort
df = df.sort_values(by=['Hemorrhage Level', 'Gender'])

# ------------------------
# Plotting function with error bands
# ------------------------
def create_plots_for_probe(probe_location):
    subset_df = df[df['Probe Location'] == probe_location]
    if subset_df.empty:
        print(f'No data available for Probe Location: {probe_location}')
        return

    subset_df = subset_df[subset_df['Time Point'].astype(str).isin(order_of_time_points)]
    hemorrhage_levels = [10, 20, 30]

    for metric in ['Avg', 'Max', 'Min']:
        fig, axes = plt.subplots(1, len(hemorrhage_levels), figsize=(desired_length, desired_width), sharey=True)
        handles, labels = [], []

        for ax, hemorrhage_level in zip(axes, hemorrhage_levels):
            level_df = subset_df[subset_df['Hemorrhage Level'] == hemorrhage_level]

            if level_df.empty:
                ax.set_title(f"{hemorrhage_level}% Hemorrhage - No Data", fontsize=LABEL_SIZE)
                continue

            numeric_columns = level_df.select_dtypes(include=['number']).columns
            mean_df = level_df.groupby(['Time Point', 'Test Group'], as_index=False)[numeric_columns].mean()
            std_df = level_df.groupby(['Time Point', 'Test Group'], as_index=False)[numeric_columns].std()

            mean_df['Time Point'] = mean_df['Time Point'].astype(int)
            std_df['Time Point'] = std_df['Time Point'].astype(int)

            for i, test_group in enumerate(level_df['Test Group'].unique()):
                mean_subset = mean_df[mean_df['Test Group'] == test_group]
                std_subset = std_df[std_df['Test Group'] == test_group]

                if mean_subset.empty:
                    continue

                color = color_palette[i % len(color_palette)]
                marker = occlusion_markers.get(test_group, 'o')

                # 🔹 Forced legend label mapping
                legend_label = occlusion_label_map.get(test_group, test_group)

                line, = ax.plot(
                    mean_subset['Time Point'],
                    mean_subset[metric],
                    marker=marker,
                    markersize=8,
                    linewidth=2,
                    label=legend_label,
                    color=color
                )

                ax.fill_between(
                    mean_subset['Time Point'],
                    mean_subset[metric] - std_subset[metric],
                    mean_subset[metric] + std_subset[metric],
                    alpha=0.2,
                    color=color
                )

                if legend_label not in labels:
                    handles.append(line)
                    labels.append(legend_label)

            ax.axhline(y=1, color="gray", linestyle="--", linewidth=1)

            # 🔹 Title now ONLY % Hemorrhage
            ax.set_title(f"{hemorrhage_level}% Hemorrhage", fontsize=LABEL_SIZE)

            # 🔹 X-axis label updated
            ax.set_xlabel("Time (mins)", fontsize=LABEL_SIZE)

            # 🔹 Custom tick spacing
            min_time = mean_df['Time Point'].min()
            max_time = mean_df['Time Point'].max()
            ax.set_xticks(np.arange(min_time, max_time + 1, X_TICK_INTERVAL))

            ax.grid(True, which='major', linestyle='--', linewidth=0.5)
            ax.tick_params(axis='both', labelsize=AXIS_TICK_SIZE)

        # 🔹 Y-axis forced text
        # axes[0].set_ylabel("Normalized Renal Flow", fontsize=LABEL_SIZE)
        # Clean probe location text
        formatted_probe = str(probe_location).replace("_", " ").strip().title()

        axes[0].set_ylabel(f"Normalized {formatted_probe}", fontsize=LABEL_SIZE)



        # 🔹 Legend with title and forced naming
        # axes[-1].legend(
        #     handles,
        #     labels,
        #     title="Intervention Group",
        #     loc="center left",
        #     bbox_to_anchor=(1.02, 0.5),
        #     fontsize=LEGEND_SIZE,
        #     title_fontsize=LEGEND_SIZE,
        #     borderaxespad=0
        # )

        plt.tight_layout()
        png_filename = os.path.join(output_dir, f'Probe_{probe_location}_{metric}.png')
        plt.savefig(png_filename, dpi=300, bbox_inches="tight")
        plt.close()

# ------------------------
# Run for all probe locations
# ------------------------
for probe_location in df['Probe Location'].unique():
    create_plots_for_probe(probe_location)


# PV - By Hemorrhage Level

In [7]:
# import os
# import pandas as pd
# import matplotlib.pyplot as plt
# import numpy as np
# import re

# AXIS_TICK_SIZE = 16
# LABEL_SIZE = 18
# LEGEND_SIZE = 14

# # 🔹 NEW: Control x-axis tick interval here
# X_TICK_INTERVAL = 5   # change to any minute spacing you want

# # Define marker symbols and colors
# marker_info = {
#     "None":    {"marker": "o", "color": "black"},
#     "Partial": {"marker": "o", "color": "blue"},
#     "Full":    {"marker": "o", "color": "red"}
# }

# legend_order = ["None", "Partial", "Full"]

# # 🔹 Forced legend label mapping
# occlusion_label_map = {
#     "None": "No Aortic Occlusion",
#     "No": "No Aortic Occlusion",
#     "Partial": "p-REBOA",
#     "Full": "f-REBOA"
# }

# # ------------------------
# # Load and preprocess data
# # ------------------------
# file_path = 'PV_Fractional_Increase_T0-1.xlsx'
# df = pd.read_excel(file_path)

# df = df.drop(columns=["Weight (kg)", "Gender"])

# df['Time Point Raw'] = df['Time Point'].astype(str)
# df = df[df['Time Point Raw'].isin(time_point_order)]

# def extract_time_point_number(time_point):
#     match = re.match(r"T(\d+)-", str(time_point))
#     return int(match.group(1)) if match else None

# df['Time Point'] = df['Time Point Raw'].apply(extract_time_point_number)

# def extract_treatment_details(treatment):
#     match = re.match(r"(\d+)%\s*(\w+)", str(treatment))
#     if match:
#         return match.group(1), match.group(2)
#     return None, None

# df[['Treatment Percentage', 'Treatment Group']] = df['Treatment Group'].apply(
#     lambda x: pd.Series(extract_treatment_details(x))
# )

# df = df.sort_values('Time Point')

# # ------------------------
# # Create output folder
# # ------------------------
# output_folder = "Pressure-Volume"
# os.makedirs(output_folder, exist_ok=True)

# # ------------------------
# # Plotting function with shaded std
# # ------------------------
# def create_panel_plots():
#     numeric_columns = df.select_dtypes(include=['number']).columns
#     hemorrhage_levels = sorted(df['Hemorrhage Level'].unique())

#     for term in [t for t in terms_to_plot if t in numeric_columns]:
#         fig, axes = plt.subplots(1, len(hemorrhage_levels), figsize=(18, 6), sharey=True)
#         if len(hemorrhage_levels) == 1:
#             axes = [axes]

#         handles, labels = [], []

#         for ax, hemorrhage_level in zip(axes, hemorrhage_levels):
#             subset_df = df[df['Hemorrhage Level'] == hemorrhage_level]

#             if subset_df.empty:
#                 ax.set_title(f"{hemorrhage_level}% Hemorrhage - No Data", fontsize=LABEL_SIZE)
#                 continue

#             mean_df = subset_df.groupby(['Treatment Group', 'Time Point'], as_index=False).mean(numeric_only=True)
#             std_df  = subset_df.groupby(['Treatment Group', 'Time Point'], as_index=False).std(numeric_only=True)

#             for treatment_key in legend_order:
#                 mean_subset = mean_df[mean_df['Treatment Group'] == treatment_key]
#                 std_subset  = std_df[std_df['Treatment Group'] == treatment_key]

#                 if mean_subset.empty:
#                     continue

#                 color = marker_info.get(treatment_key, {"color": "black"})["color"]
#                 marker = marker_info.get(treatment_key, {"marker": "o"})["marker"]

#                 # 🔹 Forced legend naming
#                 legend_label = occlusion_label_map.get(treatment_key, treatment_key)

#                 line, = ax.plot(
#                     mean_subset['Time Point'],
#                     mean_subset[term],
#                     marker=marker,
#                     markersize=8,
#                     linewidth=2,
#                     color=color,
#                     label=legend_label
#                 )

#                 ax.fill_between(
#                     mean_subset['Time Point'],
#                     mean_subset[term] - std_subset[term],
#                     mean_subset[term] + std_subset[term],
#                     alpha=0.2,
#                     color=color
#                 )

#                 if legend_label not in labels:
#                     handles.append(line)
#                     labels.append(legend_label)

#             ax.axhline(y=1, color='gray', linestyle='--', linewidth=1)

#             # 🔹 Title cropped to only % Hemorrhage
#             ax.set_title(f"{hemorrhage_level}% Hemorrhage", fontsize=LABEL_SIZE)

#             # 🔹 X-axis updated
#             ax.set_xlabel("Time (mins)", fontsize=LABEL_SIZE)

#             # 🔹 Custom tick interval
#             min_time = subset_df['Time Point'].min()
#             max_time = subset_df['Time Point'].max()
#             ax.set_xticks(np.arange(min_time, max_time + 1, X_TICK_INTERVAL))

#             # ax.grid(True, which='both', linestyle='--', linewidth=0.5)
#             ax.tick_params(axis='both', labelsize=AXIS_TICK_SIZE)

#         # 🔹 Y-axis forced
#         # axes[0].set_ylabel("Normalized Renal Flow", fontsize=LABEL_SIZE)
#         # Clean and format term name for display
#         formatted_term = str(term).replace("_", " ").strip().title()

#         axes[0].set_ylabel(f"Normalized {formatted_term}", fontsize=LABEL_SIZE)


#         # 🔹 Legend formatting
#         axes[-1].legend(
#             handles,
#             labels,
#             # title="Intervention Group",
#             loc='center left',
#             bbox_to_anchor=(1.02, 0.5),
#             fontsize=LEGEND_SIZE,
#             title_fontsize=LEGEND_SIZE,
#             borderaxespad=0
#         )

#         plt.tight_layout()
#         png_filename = os.path.join(output_folder, f'{term}_Panel.png')
#         plt.savefig(png_filename, dpi=300, bbox_inches='tight')
#         plt.close()

# # ------------------------
# # Run plotting
# # ------------------------
# create_panel_plots()





In [8]:


import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re

# ------------------------
# 🔹 GLOBAL PLOT CONFIG
# ------------------------
PLOT_CONFIG = {
    "figsize": (36, 8),
    "dpi": 300,

    # Font sizes
    "axis_tick_size": 18,
    "label_size": 20,
    "legend_size": 16,

    # X-axis control
    "x_tick_interval": 5,
    "x_min": 0,      # set None for auto
    "x_max": 240,    # set None for auto

    # Line styling
    "line_width": 2,
    "marker_size": 8,
    "fill_alpha": 0.2,

    # Reference line
    "ref_line_y": 1,


}

# ------------------------
# Style definitions
# ------------------------
marker_info = {
    "None":    {"marker": "o", "color": "black"},
    "Partial": {"marker": "o", "color": "blue"},
    "Full":    {"marker": "o", "color": "red"}
}

legend_order = ["None", "Partial", "Full"]

occlusion_label_map = {
    "None": "No Aortic Occlusion",
    "No": "No Aortic Occlusion",
    "Partial": "p-REBOA",
    "Full": "f-REBOA"
}

# ------------------------
# Load and preprocess data
# ------------------------
file_path = 'PV_Fractional_Increase_T0-1.xlsx'
df = pd.read_excel(file_path)

df = df.drop(columns=["Weight (kg)", "Gender"])

df['Time Point Raw'] = df['Time Point'].astype(str)
df = df[df['Time Point Raw'].isin(time_point_order)]

# 🔹 Custom time extraction
def extract_time_point_number(time_point):
    match = re.match(r"T(\d+)-(\d+)", str(time_point))
    if match:
        first = int(match.group(1))
        second = int(match.group(2))

        if time_point in ["T29-30", "T239-240"]:
            return second

        return first
    return None

df['Time Point'] = df['Time Point Raw'].apply(extract_time_point_number)

def extract_treatment_details(treatment):
    match = re.match(r"(\d+)%\s*(\w+)", str(treatment))
    if match:
        return match.group(1), match.group(2)
    return None, None

df[['Treatment Percentage', 'Treatment Group']] = df['Treatment Group'].apply(
    lambda x: pd.Series(extract_treatment_details(x))
)

df = df.sort_values('Time Point')

# ------------------------
# Output folder
# ------------------------
output_folder = "Pressure-Volume"
os.makedirs(output_folder, exist_ok=True)

# ------------------------
# Plotting function
# ------------------------
def create_panel_plots():
    numeric_columns = df.select_dtypes(include=['number']).columns
    hemorrhage_levels = sorted(df['Hemorrhage Level'].unique())

    for term in [t for t in terms_to_plot if t in numeric_columns]:

        fig, axes = plt.subplots(
            1,
            len(hemorrhage_levels),
            figsize=PLOT_CONFIG["figsize"],
            sharey=True
        )

        if len(hemorrhage_levels) == 1:
            axes = [axes]

        handles, labels = [], []

        for ax, hemorrhage_level in zip(axes, hemorrhage_levels):
            subset_df = df[df['Hemorrhage Level'] == hemorrhage_level]

            if subset_df.empty:
                ax.set_title(
                    f"{hemorrhage_level}% Hemorrhage - No Data",
                    fontsize=PLOT_CONFIG["label_size"]
                )
                continue

            numeric_columns = subset_df.select_dtypes(include=['number']).columns

            mean_df = subset_df.groupby(
                ['Treatment Group', 'Time Point'],
                as_index=False
            )[numeric_columns].mean()

            std_df = subset_df.groupby(
                ['Treatment Group', 'Time Point'],
                as_index=False
            )[numeric_columns].std()

            mean_df['Time Point'] = mean_df['Time Point'].astype(int)
            std_df['Time Point'] = std_df['Time Point'].astype(int)

            for treatment_key in legend_order:
                mean_subset = mean_df[mean_df['Treatment Group'] == treatment_key]
                std_subset = std_df[std_df['Treatment Group'] == treatment_key]

                if mean_subset.empty:
                    continue

                color = marker_info[treatment_key]["color"]
                marker = marker_info[treatment_key]["marker"]

                legend_label = occlusion_label_map.get(treatment_key, treatment_key)

                line, = ax.plot(
                    mean_subset['Time Point'],
                    mean_subset[term],
                    marker=marker,
                    markersize=PLOT_CONFIG["marker_size"],
                    linewidth=PLOT_CONFIG["line_width"],
                    color=color,
                    label=legend_label
                )

                ax.fill_between(
                    mean_subset['Time Point'],
                    mean_subset[term] - std_subset[term],
                    mean_subset[term] + std_subset[term],
                    alpha=PLOT_CONFIG["fill_alpha"],
                    color=color
                )

                if legend_label not in labels:
                    handles.append(line)
                    labels.append(legend_label)

            ax.axhline(
                y=PLOT_CONFIG["ref_line_y"],
                color='gray',
                linestyle='--',
                linewidth=1
            )

            ax.set_title(
                f"{hemorrhage_level}% Hemorrhage",
                fontsize=PLOT_CONFIG["label_size"]
            )

            ax.set_xlabel(
                "Time (mins)",
                fontsize=PLOT_CONFIG["label_size"]
            )

            # 🔹 X-axis ticks
            if PLOT_CONFIG["x_min"] is not None and PLOT_CONFIG["x_max"] is not None:
                start_tick = PLOT_CONFIG["x_min"]
                end_tick = PLOT_CONFIG["x_max"]
            else:
                min_time = mean_df['Time Point'].min()
                max_time = mean_df['Time Point'].max()
                start_tick = (min_time // PLOT_CONFIG["x_tick_interval"]) * PLOT_CONFIG["x_tick_interval"]
                end_tick = ((max_time // PLOT_CONFIG["x_tick_interval"]) + 1) * PLOT_CONFIG["x_tick_interval"]


            ax.minorticks_on()
            ax.grid(True, which='major', linestyle='--', linewidth=0.7)
            # ax.grid(True, which='minor', linestyle=':', linewidth=0.5)

            ax.set_xticks(np.arange(
                start_tick,
                end_tick + 1,
                PLOT_CONFIG["x_tick_interval"]
            ))

            ax.tick_params(
                axis='both',
                labelsize=PLOT_CONFIG["axis_tick_size"]
            )

        formatted_term = str(term).replace("_", " ").strip().title()

        axes[0].set_ylabel(
            f"Normalized {formatted_term}",
            fontsize=PLOT_CONFIG["label_size"]
        )



# Legend
        # axes[-1].legend(
        #     handles,
        #     labels,
        #     loc="center left",
        #     bbox_to_anchor=(1.02, 0.5),
        #     fontsize=PLOT_CONFIG["legend_size"],
        #     title_fontsize=PLOT_CONFIG["legend_size"],
        #     borderaxespad=0
        # )

        plt.tight_layout()

        png_filename = os.path.join(output_folder, f'{term}_Panel.png')
        plt.savefig(
            png_filename,
            dpi=PLOT_CONFIG["dpi"],
            bbox_inches='tight'
        )

        plt.close()

# ------------------------
# Run plotting
# ------------------------
create_panel_plots()


In [9]:
print('Done')

Done


# Box Plot PV Loop


In [10]:
time_point_order = [
    "T0-1",
    "T30-T31", 
    "T60-61",
    "T84-85",
    "T120-121",
    "T180-181",
    "T239-240"
]

# List of terms to be plotted
terms_to_plot = [
    "Stroke Work",
    "Stroke Volume", 
    "Ejection Fraction",
    "Heart Rate",
    "Cardiac Output",
    "Elastance",
    "ESV",
    "ESP",
    "EDV",
    "EDP"
]


# Hemorrhage Phase Plots

In [11]:
order_of_time_points = ["0", 
                        "6",
                        "11",
                        "16",
                        "21",
                        "26",
                        "30"]

In [12]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------
# Parameters
# ------------------------
file_path = 'Fractional_Increase_0.xlsx'

AXIS_TICK_SIZE = 16
TITLE_SIZE = 20
LABEL_SIZE = 18

color_palette = ['#E69F00', '#009E73', '#CC79A7']  # Orange, Teal, Purple


marker_map = {
    10: 'o',   # circle
    20: 'o',   # square
    30: 'o'    # triangle
}

output_dir = 'Hemorrhage Comparison'
os.makedirs(output_dir, exist_ok=True)

# ------------------------
# Load and preprocess data
# ------------------------
df = pd.read_excel(file_path)



df['Time Point'] = df['Time Point'].astype(int)
df = df[df['Time Point'].astype(str).isin(order_of_time_points)]

df['Test Group'] = df['Occlusion Group']
df = df.sort_values(by=['Hemorrhage Level', 'Gender'])

# ------------------------
# Plotting function
# ------------------------
def create_combined_plot(probe_location):
    subset_df = df[df['Probe Location'] == probe_location]
    if subset_df.empty:
        print(f'No data available for Probe Location: {probe_location}')
        return

    hemorrhage_levels = [10, 20, 30]

    for metric in ['Avg', 'Max', 'Min']:
        plt.figure(figsize=(10, 6))

        for i, hemorrhage_level in enumerate(hemorrhage_levels):
            level_df = subset_df[subset_df['Hemorrhage Level'] == hemorrhage_level]
            if level_df.empty:
                continue

            mean_df = level_df.groupby('Time Point')[metric].mean()
            std_df = level_df.groupby('Time Point')[metric].std()

            time_points = [int(tp) for tp in order_of_time_points if int(tp) in mean_df.index]
            mean_values = [mean_df[tp] for tp in time_points]
            std_values = [std_df[tp] for tp in time_points]

            plt.plot(
                time_points,
                mean_values,
                marker=marker_map[hemorrhage_level],
                markersize=8,
                linewidth=2,
                color=color_palette[i],
                label=f'{hemorrhage_level}% Hemorrhage'
            )

            plt.fill_between(
                time_points,
                [m - s for m, s in zip(mean_values, std_values)],
                [m + s for m, s in zip(mean_values, std_values)],
                color=color_palette[i],
                alpha=0.2
            )

        plt.axhline(y=1, color="gray", linestyle="--", linewidth=1)
        plt.xlabel("Time (mins)", fontsize=LABEL_SIZE)
        plt.ylabel(f"Normalized {probe_location} - {metric}", fontsize=LABEL_SIZE)
        plt.title(f"{probe_location} - {metric}", fontsize=TITLE_SIZE)
        plt.grid(True, linestyle='--', linewidth=0.5)
        plt.xticks(fontsize=AXIS_TICK_SIZE)
        plt.yticks(fontsize=AXIS_TICK_SIZE)
        plt.legend(fontsize=12)

        plt.tight_layout()
        plt.savefig(
            os.path.join(output_dir, f'Probe_{probe_location}_{metric}_combined.png'),
            dpi=300
        )
        plt.close()

# ------------------------
# Run for all probe locations
# ------------------------
for probe_location in df['Probe Location'].unique():
    create_combined_plot(probe_location)


# Box Plots for PV

In [13]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re

AXIS_TICK_SIZE = 24
TITLE_SIZE = 24
LABEL_SIZE = 24
LEGEND_SIZE = 22

# Control x-axis tick interval here
X_TICK_INTERVAL = 1   # spacing in index units (keep as is since boxes are equidistant in these plots)

marker_info = {
    "None":    {"color": "black"},
    "Partial": {"color": "blue"},
    "Full":    {"color": "red"}
}


# marker_info = {
#     "None":    {"color":"#E69F00"},   # Orange
#     "Partial": {"color": "#009E73"},   # Teal/Green
#     "Full":    {"color": "#CC79A7"}    # Purple/Magenta
# }


legend_order = ["None", "Partial", "Full"]

# Forced legend naming
occlusion_label_map = {
    "None": "No Aortic Occlusion",
    "No": "No Aortic Occlusion",
    "Partial": "p-REBOA",
    "Full": "f-REBOA"
}

# ------------------------
# User-specified time points
# ------------------------
time_point_order = [
    "T30-T31",
    "T60-61",
    "T75-76",
    "T85-86",
    "T120-121",
    "T180-181",
    "T239-240"
]

time_value_map = {
    "T30-T31": 30,
    "T60-61": 60,
    "T75-76": 75,
    "T85-86": 85,
    "T120-121": 120,
    "T180-181": 180,
    "T239-240": 240
}

time_index_map = {tp: i for i, tp in enumerate(time_point_order)}

# ------------------------
# Load and preprocess data
# ------------------------
file_path = 'PV_Fractional_Increase_T0-1.xlsx'
df = pd.read_excel(file_path)
df = df.drop(columns=["Weight (kg)", "Gender"])
df['Time Point Raw'] = df['Time Point'].astype(str)

df = df[df['Time Point Raw'].isin(time_point_order)]
df['Time Index'] = df['Time Point Raw'].map(time_index_map)

def extract_treatment_details(treatment):
    match = re.match(r"(\d+)%\s*(\w+)", str(treatment))
    if match:
        return match.group(1), match.group(2)
    return None, None

df[['Treatment Percentage', 'Treatment Group']] = df['Treatment Group'].apply(
    lambda x: pd.Series(extract_treatment_details(x))
)

df = df.sort_values('Time Index')

# ------------------------
# Create output folder
# ------------------------
output_folder = "PV Box Plots"
os.makedirs(output_folder, exist_ok=True)

# ------------------------
# Plotting function
# ------------------------
def create_panel_plots():
    numeric_columns = df.select_dtypes(include=['number']).columns
    hemorrhage_levels = sorted(df['Hemorrhage Level'].unique())

    for term in [t for t in terms_to_plot if t in numeric_columns]:
        fig, axes = plt.subplots(
            1,
            len(hemorrhage_levels),
            figsize=(36, 6),
            sharey=True
        )

        if len(hemorrhage_levels) == 1:
            axes = [axes]

        handles, labels = [], []

        for ax, hemorrhage_level in zip(axes, hemorrhage_levels):
            subset_df = df[df['Hemorrhage Level'] == hemorrhage_level]

            if subset_df.empty:
                ax.set_title(f"{hemorrhage_level}% Hemorrhage - No Data", fontsize=TITLE_SIZE)
                continue

            n_treatments = len(legend_order)
            box_width = 0.2
            offsets = [(i - (n_treatments-1)/2) * box_width*1.5 for i in range(n_treatments)]

            for i, treatment_key in enumerate(legend_order):
                treatment_df = subset_df[subset_df['Treatment Group'] == treatment_key]
                if treatment_df.empty:
                    continue

                color = marker_info[treatment_key]["color"]
                legend_label = occlusion_label_map.get(treatment_key, treatment_key)

                data, positions = [], []

                for tp in time_point_order:
                    vals = treatment_df[treatment_df['Time Point Raw'] == tp][term].dropna()
                    if not vals.empty:
                        data.append(vals)
                        positions.append(time_index_map[tp] + offsets[i])

                if not data:
                    continue

                bp = ax.boxplot(
                    data,
                    positions=positions,
                    widths=box_width,
                    patch_artist=True,
                    showfliers=False
                )

                for box in bp['boxes']:
                    box.set_facecolor(color)
                    box.set_alpha(0.6)
                    box.set_linewidth(2)

                for element in ['whiskers', 'caps', 'medians']:
                    for item in bp[element]:
                        item.set_linewidth(2)

                if legend_label not in labels:
                    handles.append(bp['boxes'][0])
                    labels.append(legend_label)

            ax.axhline(y=1, color='gray', linestyle='--', linewidth=1)

            # 🔹 Title cleaned
            ax.set_title(f"{hemorrhage_level}% Hemorrhage", fontsize=TITLE_SIZE)

            # 🔹 Custom tick interval
            xtick_positions = np.arange(
                0,
                len(time_point_order),
                X_TICK_INTERVAL
            )

            ax.set_xticks(xtick_positions)
            ax.set_xticklabels(
                [time_value_map[time_point_order[int(i)]] for i in xtick_positions]
            )

            ax.set_xlabel("Time (mins)", fontsize=LABEL_SIZE)

            # ax.grid(True, axis='y', linestyle='--', linewidth=0.5)
            ax.tick_params(axis='both', labelsize=AXIS_TICK_SIZE)

        # 🔹 Y-axis forced
        # axes[0].set_ylabel("Normalized Renal Flow", fontsize=LABEL_SIZE)
        # Automatically clean and format term name
        formatted_term = term.replace("_", " ")

        axes[0].set_ylabel(f"Normalized {formatted_term}", fontsize=LABEL_SIZE)


        # 🔹 Legend standardized
        axes[-1].legend(
            handles,
            labels,
            title="Intervention Type",
            loc='center left',
            bbox_to_anchor=(1.02, 0.5),
            fontsize=LEGEND_SIZE,
            title_fontsize=LEGEND_SIZE,
            borderaxespad=0
        )

        plt.tight_layout()
        plt.savefig(
            os.path.join(output_folder, f"{term}_Panel.png"),
            dpi=300,
            bbox_inches='tight'
        )
        plt.close()

# ------------------------
# Run plotting
# ------------------------
create_panel_plots()


# TEG


In [14]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------
# Parameters
# ------------------------
file_path = 'ERNE_TEG_cleaned.xlsx'
teg_metrics = ['R_time', 'K_time', 'Angle', 'MA', 'G', 'LY30']

AXIS_TICK_SIZE = 16
TITLE_SIZE = 20
LABEL_SIZE = 18

# Marker symbols and colors
marker_info = {
    "No": {"marker": "o", "color": "black"},
    "Partial": {"marker": "o", "color": "blue"},
    "Full": {"marker": "o", "color": "red"}
}
legend_order = ["No", "Partial", "Full"]  # ensure consistent legend order

# Output directory
output_dir = 'TEG'
os.makedirs(output_dir, exist_ok=True)

# ------------------------
# Load and preprocess data
# ------------------------
df = pd.read_excel(file_path)

time_col = 'Time Point'
group_col = 'Occlusion Group'
level_col = 'Hemorrhage Level'

# Ensure numeric
df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
df = df.dropna(subset=[time_col])
df[time_col] = df[time_col].astype(float)

# Correct "None Occlusion" → "No Occlusion"
df[group_col] = df[group_col].replace({"None Occlusion": "No Occlusion"})

# Map group to simplified keys for marker/color
def map_group_key(name):
    if "No" in name:
        return "No"
    elif "Partial" in name:
        return "Partial"
    elif "Full" in name:
        return "Full"
    return "No"

df['Group Key'] = df[group_col].apply(map_group_key)

# ------------------------
# Plotting function
# ------------------------
def plot_teg_panels():
    hemorrhage_levels = sorted(df[level_col].unique())

    for metric in teg_metrics:
        fig, axes = plt.subplots(1, len(hemorrhage_levels), figsize=(18, 6), sharey=True)
        if len(hemorrhage_levels) == 1:
            axes = [axes]

        handles, labels = [], []

        for ax, hemorrhage_level in zip(axes, hemorrhage_levels):
            level_df = df[df[level_col] == hemorrhage_level]

            # Group by Occlusion Group & Time Point
            grouped_df = level_df.groupby(['Group Key', time_col])[metric].agg(['mean', 'std']).reset_index()

            for key in legend_order:  # enforce legend order
                subset = grouped_df[grouped_df['Group Key'] == key]
                if subset.empty:
                    continue

                label_name = f"{key} Occlusion"

                line, = ax.plot(
                    subset[time_col],
                    subset['mean'],
                    marker=marker_info[key]["marker"],
                    markersize=8,
                    linewidth=2,
                    color=marker_info[key]["color"],
                    label=label_name
                )
                if label_name not in labels:
                    handles.append(line)
                    labels.append(label_name)

                # Error bars
                ax.fill_between(
                    subset[time_col],
                    subset['mean'] - subset['std'],
                    subset['mean'] + subset['std'],
                    alpha=0.2,
                    color=marker_info[key]["color"]
                )

            # Reference line at y=1
            ax.axhline(y=1, color='gray', linestyle='--', linewidth=1)

            ax.set_title(f"{hemorrhage_level}% Hemorrhage - {metric}", fontsize=TITLE_SIZE)
            ax.set_xlabel("Time", fontsize=LABEL_SIZE)


            # Set x-axis ticks at intervals of 30
            xmin = df[time_col].min()
            xmax = df[time_col].max()
            ax.set_xticks(np.arange(0, xmax + 30, 30))

            ax.grid(True, which='both', linestyle='--', linewidth=0.5)
            ax.tick_params(axis='both', labelsize=AXIS_TICK_SIZE)

        axes[0].set_ylabel(metric, fontsize=LABEL_SIZE)

        # Legend next to right-most panel
        axes[-1].legend(
            handles, labels,
            loc='center left',
            bbox_to_anchor=(1.06, 0.75),
            fontsize=12,
            borderaxespad=0
        )

        plt.tight_layout()
        png_file = os.path.join(output_dir, f'TEG_{metric}_Panel.png')
        plt.savefig(png_file, dpi=300, bbox_inches='tight')
        plt.close()

# ------------------------
# Run plotting
# ------------------------
plot_teg_panels()


# CHEM 8 and CG4

In [15]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ==========================================================
# FILES
# ==========================================================

data_file = "CG4 and CHEM8 Data.xlsx"
tracker_file = "Experiment Tracker.xlsx"

output_dir = "CG4 and CHEM8 Plots"
os.makedirs(output_dir, exist_ok=True)

# ==========================================================
# TIME POINT CONTROL  (NUMERIC)
# ==========================================================

order_of_time_points = [
    0, 30, 60, 85, 120, 180, 240
]

# Control tick spacing here
X_TICK_INTERVAL = 30

# ==========================================================
# VISUAL SETTINGS
# ==========================================================

AXIS_TICK_SIZE = 16
LABEL_SIZE = 18
LEGEND_SIZE = 14

color_palette = [
    'black', 'blue', 'red',
    'purple', 'orange',
    'cyan', 'magenta'
]

occlusion_label_map = {
    'No Occlusion': 'No Aortic Occlusion',
    'Partial Occlusion': 'p-REBOA',
    'Full Occlusion': 'f-REBOA'
}

occlusion_order = [
    'No Occlusion',
    'Partial Occlusion',
    'Full Occlusion'
]

# ==========================================================
# METRIC RENAME MAPPING  (EDIT HERE)
# ==========================================================

metric_name_map = {
    "ph": "pH (unitless)",
    "tco2_x": "TCO₂ (CG4) (mmol/L)",
    "po2": "pO₂ (mmHg)",
    "pco2": "pCO₂ (mmHg)",
    "be_ecf": "Base Excess (ECF) (mmol/L)",
    "hco3": "HCO₃⁻ (mmol/L)",
    "so2": "O₂ Saturation (%)",
    "lac": "Lactate (mmol/L)",
    "temp": "Temperature (°C)",
    "temp_ph": "Temperature Corrected pH (unitless)",
    "temp_po2": "Temp Corrected pO₂ (mmHg)",
    "temp_pco2": "Temp Corrected pCO₂ (mmHg)",
    "cl": "Chloride (mmol/L)",
    "k": "Potassium (mmol/L)",
    "na": "Sodium (mmol/L)",
    "bun": "BUN (mg/dL)",
    "ica": "Ionized Calcium (mmol/L)",
    "glu": "Glucose (mg/dL)",
    "tco2_y": "TCO₂ (CHEM8) (mmol/L)",
    "angap": "Anion Gap (mmol/L)",
    "hb": "Hemoglobin (g/dL)",
    "hct": "Hematocrit (%)",
    "crea": "Creatinine (mg/dL)"
}

# ==========================================================
# LOAD TRACKER DATA
# ==========================================================

tracker = pd.read_excel(
    tracker_file,
    sheet_name=['T30 categorization', 'T60 categorization']
)

group_10 = set(tracker['T30 categorization']['Group_10'].dropna())
group_20 = set(tracker['T30 categorization']['Group_20'].dropna())
group_30 = set(tracker['T30 categorization']['Group_30'].dropna())

full_occlusion = set(tracker['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(tracker['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(tracker['T60 categorization']['No_Occlusion'].dropna())

# ==========================================================
# LOAD ALL SUBJECT SHEETS
# ==========================================================

all_sheets = pd.read_excel(data_file, sheet_name=None)

combined_df = []

for sheet_name, sheet_df in all_sheets.items():

    if 'name' in sheet_df.columns:
        sheet_df = sheet_df.drop(columns=['name'])

    sheet_df['Subject'] = sheet_df['Subject'].astype(str)

    def assign_hemorrhage(subj):
        if subj in group_10:
            return 10
        elif subj in group_20:
            return 20
        elif subj in group_30:
            return 30
        else:
            return np.nan

    def assign_occlusion(subj):
        if subj in full_occlusion:
            return 'Full Occlusion'
        elif subj in partial_occlusion:
            return 'Partial Occlusion'
        elif subj in no_occlusion:
            return 'No Occlusion'
        else:
            return np.nan

    sheet_df['Hemorrhage Level'] = sheet_df['Subject'].apply(assign_hemorrhage)
    sheet_df['Occlusion Group'] = sheet_df['Subject'].apply(assign_occlusion)

    combined_df.append(sheet_df)

df = pd.concat(combined_df, ignore_index=True)

# Remove subjects not in tracker
df = df.dropna(subset=['Hemorrhage Level', 'Occlusion Group'])

# ==========================================================
# FILTER TIME POINTS (NUMERIC SAFE)
# ==========================================================

df['Time Point'] = pd.to_numeric(df['Time Point'], errors='coerce')
df = df[df['Time Point'].isin(order_of_time_points)]

df = df.sort_values('Time Point')

# ==========================================================
# IDENTIFY METRICS
# ==========================================================

exclude_cols = [
    'Subject',
    'Time Point',
    'Hemorrhage Level',
    'Occlusion Group'
]

numeric_columns = df.select_dtypes(include=['number']).columns
metrics = [col for col in numeric_columns if col not in exclude_cols]

# ==========================================================
# PLOTTING FUNCTION
# ==========================================================

def create_plots_for_metric(metric):

    hemorrhage_levels = [10, 20, 30]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

    handles, labels = [], []

    for ax, hemorrhage_level in zip(axes, hemorrhage_levels):

        level_df = df[df['Hemorrhage Level'] == hemorrhage_level]

        if level_df.empty:
            ax.set_title(f"{hemorrhage_level}% Hemorrhage - No Data",
                         fontsize=LABEL_SIZE)
            continue

        mean_df = level_df.groupby(
            ['Time Point', 'Occlusion Group'],
            as_index=False
        )[metric].mean()

        std_df = level_df.groupby(
            ['Time Point', 'Occlusion Group'],
            as_index=False
        )[metric].std()

        for i, occlusion in enumerate(occlusion_order):

            if occlusion not in mean_df['Occlusion Group'].unique():
                continue

            mean_subset = mean_df[mean_df['Occlusion Group'] == occlusion]
            std_subset = std_df[std_df['Occlusion Group'] == occlusion]

            color = color_palette[i % len(color_palette)]
            legend_label = occlusion_label_map.get(occlusion, occlusion)

            x_vals = mean_subset['Time Point']

            line, = ax.plot(
                x_vals,
                mean_subset[metric],
                marker='o',
                markersize=8,
                linewidth=2,
                label=legend_label,
                color=color
            )

            ax.fill_between(
                x_vals,
                mean_subset[metric] - std_subset[metric],
                mean_subset[metric] + std_subset[metric],
                alpha=0.2,
                color=color
            )

            if legend_label not in labels:
                handles.append(line)
                labels.append(legend_label)

        ax.set_title(f"{hemorrhage_level}% Hemorrhage",
                     fontsize=LABEL_SIZE)

        ax.set_xlabel("Time (mins)", fontsize=LABEL_SIZE)

        ax.set_xticks(
            np.arange(
                min(order_of_time_points),
                max(order_of_time_points) + 1,
                X_TICK_INTERVAL
            )
        )

        # ax.grid(True, which='both', linestyle='--', linewidth=0.5)
        ax.tick_params(axis='both', labelsize=AXIS_TICK_SIZE)

    y_label = metric_name_map.get(metric, metric)
    axes[0].set_ylabel(y_label, fontsize=LABEL_SIZE)

    axes[-1].legend(
        handles,
        labels,
        title="Intervention Group",
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        fontsize=LEGEND_SIZE,
        title_fontsize=LEGEND_SIZE
    )

    plt.tight_layout()
    plt.savefig(
        os.path.join(output_dir, f"{metric}.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

# ==========================================================
# RUN ALL METRICS
# ==========================================================

for metric in metrics:
    create_plots_for_metric(metric)


In [16]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator

# ==========================================================
# FILES
# ==========================================================

data_file = "CG4 and CHEM8 Data.xlsx"
tracker_file = "Experiment Tracker.xlsx"

output_dir = "CG4 and CHEM8 Plots"
os.makedirs(output_dir, exist_ok=True)

plt.rcParams['axes.axisbelow'] = True  # grid behind lines

# ==========================================================
# TIME POINT CONTROL  (NUMERIC)
# ==========================================================

order_of_time_points = [
    0, 30, 60, 85, 120, 180, 240
]

# Control tick spacing here
X_TICK_INTERVAL = 30

# ==========================================================
# VISUAL SETTINGS
# ==========================================================

AXIS_TICK_SIZE = 16
LABEL_SIZE = 18
LEGEND_SIZE = 14

color_palette = [
    'black', 'blue', 'red',
    'purple', 'orange',
    'cyan', 'magenta'
]

occlusion_label_map = {
    'No Occlusion': 'No REBOA',
    'Partial Occlusion': 'p-REBOA',
    'Full Occlusion': 'f-REBOA'
}

occlusion_order = [
    'No Occlusion',
    'Partial Occlusion',
    'Full Occlusion'
]

# ==========================================================
# METRIC RENAME MAPPING
# ==========================================================

metric_name_map = {
    "ph": "pH (unitless)",
    "tco2_x": "TCO₂ (CG4) (mmol/L)",
    "po2": "pO₂ (mmHg)",
    "pco2": "pCO₂ (mmHg)",
    "be_ecf": "Base Excess (ECF) (mmol/L)",
    "hco3": "HCO₃⁻ (mmol/L)",
    "so2": "O₂ Saturation (%)",
    "lac": "Lactate (mmol/L)",
    "temp": "Temperature (°C)",
    "temp_ph": "Temperature Corrected pH (unitless)",
    "temp_po2": "Temp Corrected pO₂ (mmHg)",
    "temp_pco2": "Temp Corrected pCO₂ (mmHg)",
    "cl": "Chloride (mmol/L)",
    "k": "Potassium (mmol/L)",
    "na": "Sodium (mmol/L)",
    "bun": "BUN (mg/dL)",
    "ica": "Ionized Calcium (mmol/L)",
    "glu": "Glucose (mg/dL)",
    "tco2_y": "TCO₂ (CHEM8) (mmol/L)",
    "angap": "Anion Gap (mmol/L)",
    "hb": "Hemoglobin (g/dL)",
    "hct": "Hematocrit (%)",
    "crea": "Creatinine (mg/dL)"
}

# ==========================================================
# LOAD TRACKER DATA
# ==========================================================

tracker = pd.read_excel(
    tracker_file,
    sheet_name=['T30 categorization', 'T60 categorization']
)

group_10 = set(tracker['T30 categorization']['Group_10'].dropna())
group_20 = set(tracker['T30 categorization']['Group_20'].dropna())
group_30 = set(tracker['T30 categorization']['Group_30'].dropna())

full_occlusion = set(tracker['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(tracker['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(tracker['T60 categorization']['No_Occlusion'].dropna())

# ==========================================================
# LOAD ALL SUBJECT SHEETS
# ==========================================================

all_sheets = pd.read_excel(data_file, sheet_name=None)

combined_df = []

for sheet_name, sheet_df in all_sheets.items():

    if 'name' in sheet_df.columns:
        sheet_df = sheet_df.drop(columns=['name'])

    sheet_df['Subject'] = sheet_df['Subject'].astype(str)

    def assign_hemorrhage(subj):
        if subj in group_10:
            return 10
        elif subj in group_20:
            return 20
        elif subj in group_30:
            return 30
        else:
            return np.nan

    def assign_occlusion(subj):
        if subj in full_occlusion:
            return 'Full Occlusion'
        elif subj in partial_occlusion:
            return 'Partial Occlusion'
        elif subj in no_occlusion:
            return 'No Occlusion'
        else:
            return np.nan

    sheet_df['Hemorrhage Level'] = sheet_df['Subject'].apply(assign_hemorrhage)
    sheet_df['Occlusion Group'] = sheet_df['Subject'].apply(assign_occlusion)

    combined_df.append(sheet_df)

df = pd.concat(combined_df, ignore_index=True)

# Remove subjects not in tracker
df = df.dropna(subset=['Hemorrhage Level', 'Occlusion Group'])

# ==========================================================
# FILTER TIME POINTS
# ==========================================================

df['Time Point'] = pd.to_numeric(df['Time Point'], errors='coerce')
df = df[df['Time Point'].isin(order_of_time_points)]
df = df.sort_values('Time Point')

# ==========================================================
# IDENTIFY METRICS
# ==========================================================

exclude_cols = [
    'Subject',
    'Time Point',
    'Hemorrhage Level',
    'Occlusion Group'
]

numeric_columns = df.select_dtypes(include=['number']).columns
metrics = [col for col in numeric_columns if col not in exclude_cols]

# ==========================================================
# PLOTTING FUNCTION
# ==========================================================

def create_plots_for_metric(metric):

    hemorrhage_levels = [10, 20, 30]
    fig, axes = plt.subplots(1, 3, figsize=(30, 8), sharey=True)

    handles, labels = [], []

    for ax, hemorrhage_level in zip(axes, hemorrhage_levels):

        level_df = df[df['Hemorrhage Level'] == hemorrhage_level]

        if level_df.empty:
            ax.set_title(f"{hemorrhage_level}% Hemorrhage - No Data",
                         fontsize=LABEL_SIZE)
            continue

        mean_df = level_df.groupby(
            ['Time Point', 'Occlusion Group'],
            as_index=False
        )[metric].mean()

        std_df = level_df.groupby(
            ['Time Point', 'Occlusion Group'],
            as_index=False
        )[metric].std()

        for i, occlusion in enumerate(occlusion_order):

            if occlusion not in mean_df['Occlusion Group'].unique():
                continue

            mean_subset = mean_df[mean_df['Occlusion Group'] == occlusion]
            std_subset = std_df[std_df['Occlusion Group'] == occlusion]

            color = color_palette[i % len(color_palette)]
            legend_label = occlusion_label_map.get(occlusion, occlusion)

            x_vals = mean_subset['Time Point']

            line, = ax.plot(
                x_vals,
                mean_subset[metric],
                marker='o',
                markersize=8,
                linewidth=2,
                label=legend_label,
                color=color
            )

            ax.fill_between(
                x_vals,
                mean_subset[metric] - std_subset[metric],
                mean_subset[metric] + std_subset[metric],
                alpha=0.2,
                color=color
            )

            if legend_label not in labels:
                handles.append(line)
                labels.append(legend_label)

        ax.set_title(f"{hemorrhage_level}% Hemorrhage",
                     fontsize=LABEL_SIZE)

        ax.set_xlabel("Time (mins)", fontsize=LABEL_SIZE)

        # Major ticks
        ax.set_xticks(
            np.arange(
                min(order_of_time_points),
                max(order_of_time_points) + 1,
                X_TICK_INTERVAL
            )
        )

        # Minor ticks
        ax.minorticks_on()
        ax.xaxis.set_minor_locator(AutoMinorLocator(2))
        ax.yaxis.set_minor_locator(AutoMinorLocator(2))

        # Major grid
        ax.grid(True, which='major', linestyle='-', linewidth=0.7, alpha=0.6)

        # Minor grid
        ax.grid(True, which='minor', linestyle='--', linewidth=0.4, alpha=0.4)

        ax.tick_params(axis='both', labelsize=AXIS_TICK_SIZE)

    y_label = metric_name_map.get(metric, metric)
    axes[0].set_ylabel(y_label, fontsize=LABEL_SIZE)

    # axes[-1].legend(
    #     handles,
    #     labels,
    #     # title="Intervention Group",
    #     loc="center left",
    #     bbox_to_anchor=(1.02, 0.5),
    #     fontsize=LEGEND_SIZE,
    #     title_fontsize=LEGEND_SIZE
    # )

    plt.tight_layout()

    plt.savefig(
        os.path.join(output_dir, f"{metric}.png"),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

# ==========================================================
# RUN ALL METRICS
# ==========================================================

for metric in metrics:
    create_plots_for_metric(metric)